In [ ]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Seed Points Sales Prediction
# MAGIC
# MAGIC Predicts sales for Little Caesars expansion locations using demographic and spatial features.
# MAGIC Selects top 25% performers for expansion recommendation.

%md
## Parameters

In [ ]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "jdub_demo_aws")
dbutils.widgets.text("gold_schema", "geo_gold")

catalog = dbutils.widgets.get("catalog")
gold_schema = dbutils.widgets.get("gold_schema")

seed_points_table = f"{catalog}.{gold_schema}.lce_trade_area_features"
output_table = f"{catalog}.{gold_schema}.lce_expansion_candidates"

print(f"Input: {seed_points_table}")
print(f"Output: {output_table}")

%md
## Load Seed Points Trade Area Features

In [ ]:
seed_points = spark.table(seed_points_table)

# Add store_type if missing (backward compatibility)
if 'store_type' not in seed_points.columns:
    seed_points = seed_points.withColumn('store_type', F.lit('Little Caesars'))
    print("Added default store_type: 'Little Caesars'")
    # Materialize the change by selecting all columns explicitly
    seed_points = seed_points.select("*")

# Also handle city and state columns if missing (sometimes dropped in aggregation)
if 'city' not in seed_points.columns:
    seed_points = seed_points.withColumn('city', F.lit('Unknown'))
if 'state' not in seed_points.columns:
    seed_points = seed_points.withColumn('state', F.lit('MA'))

print(f"Total seed points: {seed_points.count()}")
display(seed_points.limit(5))

%md
## Predict Sales

Sales prediction formula for Little Caesars:
- Young adults (ages 15-34) - key demographic for quick-service pizza
- Working age adults (ages 35-54) - family demographic
- POI density (vibrant neighborhoods) - food/drink, retail, leisure
- Population in trade area

In [ ]:
# Apply sales prediction formula for Little Caesars using CARTO features
# Note: Using only demographic age buckets and POI counts (aggregated columns available in trade area features)
seed_points_with_sales = seed_points.withColumn(
    "predicted_annual_sales",
    (
        # Base: 300k
        300000 +

        # Young adults (15-34) - CARTO has 5-year age buckets
        # Using 15-19 + 20-24 + 25-29 + 30-34 as proxy for young adults
        (F.coalesce(F.col("male_15_to_19"), F.lit(0)) +
         F.coalesce(F.col("female_15_to_19"), F.lit(0)) +
         F.coalesce(F.col("male_20_to_24"), F.lit(0)) +
         F.coalesce(F.col("female_20_to_24"), F.lit(0)) +
         F.coalesce(F.col("male_25_to_29"), F.lit(0)) +
         F.coalesce(F.col("female_25_to_29"), F.lit(0)) +
         F.coalesce(F.col("male_30_to_34"), F.lit(0)) +
         F.coalesce(F.col("female_30_to_34"), F.lit(0))) * 30 +

        # Working age adults (35-54) - Family demographic
        (F.coalesce(F.col("male_35_to_39"), F.lit(0)) +
         F.coalesce(F.col("female_35_to_39"), F.lit(0)) +
         F.coalesce(F.col("male_40_to_44"), F.lit(0)) +
         F.coalesce(F.col("female_40_to_44"), F.lit(0)) +
         F.coalesce(F.col("male_45_to_49"), F.lit(0)) +
         F.coalesce(F.col("female_45_to_49"), F.lit(0)) +
         F.coalesce(F.col("male_50_to_54"), F.lit(0)) +
         F.coalesce(F.col("female_50_to_54"), F.lit(0))) * 20 +

        # POI impact using aggregated CARTO categories (note: columns are total_X_pois after aggregation)
        (F.coalesce(F.col("total_food_drink_pois"), F.lit(0)) * 200 +  # Food/drink POIs highly relevant for LCE
         F.coalesce(F.col("total_retail_pois"), F.lit(0)) * 100 +
         F.coalesce(F.col("total_leisure_pois"), F.lit(0)) * 150 +
         F.coalesce(F.col("total_poi_count"), F.lit(0)) * 50) +  # General vibrancy bonus

        # Population bonus (using CARTO population column)
        F.coalesce(F.col("population"), F.lit(0)) * 2
    ).cast("long")
).withColumn(
    "predicted_monthly_sales",
    (F.col("predicted_annual_sales") / 12).cast("long")
)

print(f"Sales predictions generated for {seed_points_with_sales.count()} seed points")
display(seed_points_with_sales.select(
    "store_number",
    "city",
    "population",
    "total_poi_count",
    "predicted_annual_sales",
    "predicted_monthly_sales"
).orderBy(F.desc("predicted_annual_sales")).limit(10))

%md
## Select Top 25%

In [ ]:
# Calculate 75th percentile threshold
percentile_75 = seed_points_with_sales.selectExpr(
    "percentile_approx(predicted_annual_sales, 0.75) as p75"
).collect()[0]['p75']

print(f"75th percentile threshold: ${percentile_75:,}")

# Filter to top 25%
top_25_percent = seed_points_with_sales.filter(
    F.col("predicted_annual_sales") >= percentile_75
)

top_count = top_25_percent.count()
total_count = seed_points_with_sales.count()
print(f"Top 25%: {top_count} locations out of {total_count} total")

display(top_25_percent.select(
    "store_number",
    "latitude",
    "longitude",
    "city",
    "state",
    "population",
    "total_poi_count",
    "predicted_annual_sales",
    "predicted_monthly_sales"
).orderBy(F.desc("predicted_annual_sales")))

%md
## Write to Gold

In [ ]:
# Add processing timestamp
top_25_final = top_25_percent.withColumn("processing_timestamp", F.current_timestamp())

# Write to gold layer
(
    top_25_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(output_table)
)

print(f"\n✓ Written {top_count} top performing seed points to {output_table}")

%md
## Summary Statistics

In [ ]:
display(spark.sql(f"""
  SELECT
    COUNT(*) as total_locations,
    ROUND(AVG(predicted_annual_sales), 0) as avg_predicted_sales,
    ROUND(MIN(predicted_annual_sales), 0) as min_predicted_sales,
    ROUND(MAX(predicted_annual_sales), 0) as max_predicted_sales,
    ROUND(AVG(population), 0) as avg_population,
    ROUND(AVG(total_poi_count), 0) as avg_poi_count
  FROM {output_table}
"""))